# Merge Column of Soil, Climate, Weather and Plant Dataset
**Student:** Mahinur Akhter &nbsp;|&nbsp; **ID:** 22201100  
**Task:** Merge 10 tabular datasets (Climate + Soil + Environmental) and 5 image datasets (Plant Disease) to build a Crop Yield Prediction ML model and a Plant Disease CNN.

---
### Pipeline Overview
1. Data Merging — 7-Step Strategy  
2. Exploratory Data Analysis (EDA)  
3. Regression Models (yield_hgha prediction)  
4. Classification Models (High / Medium / Low yield)  
5. CNN — Plant Disease Detection with Grad-CAM  

In [ ]:
import sys, os
sys.path.insert(0, 'src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Libraries loaded ✓')

---
## Part 1 — Data Merging (7 Steps)

### Dataset Overview
| # | Dataset | Type | Years | Rows | Source |
|---|---------|------|-------|------|--------|
| 1 | Crop Yield (EDA + Viz) FAO | Soil + Climate | 1961–2016 | ~56,717 | Kaggle |
| 2 | Crop Recommendation (N,P,K,pH) | Soil | 2020 | 2,200 | Kaggle |
| 3 | Crop and Soil DataSet | Soil + Climate | 2010–2023 | 8,000 | Kaggle |
| 4 | Agricultural Land Suitability Bangladesh | Soil + Climate | 2020–2024 | ~9.1M | Kaggle |
| 5 | Bangladesh Agroclimatic Crop Yield 2000-2024 | Climate + Crop | 2000–2024 | 150 | Kaggle |
| 6 | Climate Change Earth Surface Temperature | Climate | 1901–2015 | ~39,900 | Kaggle |
| 7 | Climate Data for Bangladesh 2021–2024 | Climate | 2021–2024 | ~1,460 | Kaggle |
| 8 | Bangladesh Weather Dataset 1901–2023 | Climate | 1901–2023 | ~1,386 | Kaggle |
| 9 | Environmental Sensor Telemetry Data | Environmental | ~10 yrs | 405,184 | Kaggle |
| 10 | Dhaka Air Quality 2000–2025 | Environmental | 2000–2025 | 225,000 | Kaggle |

In [ ]:
from data_merge import (
    _make_climate_df, _make_soil_df, _make_env_df,
    step1_standardize, step2_filter,
    step3_merge_climate, step4_merge_soil, step5_merge_env,
    step6_final_join, step7_handle_missing, step8_feature_engineering,
    CLIMATE_RENAME, SOIL_RENAME, ENV_RENAME,
    run_full_pipeline
)

# Generate synthetic datasets (replace with real CSVs if available)
climate_dfs_raw = [_make_climate_df(3000, 42), _make_climate_df(2500, 52)]
soil_dfs_raw    = [_make_soil_df(2500, 43),    _make_soil_df(2000, 53)]
env_dfs_raw     = [_make_env_df(2500, 44),     _make_env_df(2000, 54)]

print('Sample Climate dataset:')
display(climate_dfs_raw[0].head(3))
print('\nSample Soil dataset:')
display(soil_dfs_raw[0].head(3))
print('\nSample Environmental dataset:')
display(env_dfs_raw[0].head(3))

In [ ]:
# ── STEP 1 & 2: Standardize + Filter ─────────────────────────────────────────
print('Step 1-2: Standardize column names and filter year 2000-2024')

climate_dfs = [step2_filter(step1_standardize(d, CLIMATE_RENAME)) for d in climate_dfs_raw]
soil_dfs    = [step2_filter(step1_standardize(d, SOIL_RENAME))    for d in soil_dfs_raw]
env_dfs     = [step2_filter(step1_standardize(d, ENV_RENAME))     for d in env_dfs_raw]

print(f'  Climate datasets: {[len(d) for d in climate_dfs]} rows')
print(f'  Soil datasets:    {[len(d) for d in soil_dfs]} rows')
print(f'  Env datasets:     {[len(d) for d in env_dfs]} rows')

In [ ]:
# ── STEP 3: Merge Climate datasets ───────────────────────────────────────────
print('Step 3: Merge Climate datasets on (Year, Country)')
df_climate = step3_merge_climate(climate_dfs)
print(f'  Shape: {df_climate.shape}')
display(df_climate.head(4))

In [ ]:
# ── STEP 4: Merge Soil datasets ───────────────────────────────────────────────
print('Step 4: Merge Soil datasets on (Year, Country, Crop_Type)')
df_soil = step4_merge_soil(soil_dfs)
print(f'  Shape: {df_soil.shape}')
display(df_soil.head(4))

In [ ]:
# ── STEP 5: Merge Environmental datasets ─────────────────────────────────────
print('Step 5: Merge Environmental datasets on (Year, Country)')
df_env = step5_merge_env(env_dfs)
print(f'  Shape: {df_env.shape}')
display(df_env.head(4))

In [ ]:
# ── STEP 6: Final Left Join ───────────────────────────────────────────────────
print('Step 6: Final Left Join → Soil ← Climate ← Environmental')
df_merged = step6_final_join(df_climate, df_soil, df_env)
print(f'  Shape: {df_merged.shape}')
display(df_merged.head(4))

In [ ]:
# ── STEP 7: Handle Missing Values ────────────────────────────────────────────
print('Step 7: Handle missing — median for numeric, mode for categorical')
print(f'  Missing before: {df_merged.isnull().sum().sum()}')
df_clean = step7_handle_missing(df_merged)
print(f'  Missing after:  {df_clean.isnull().sum().sum()}')
print(f'  Shape: {df_clean.shape}')

In [ ]:
# ── STEP 8: Feature Engineering ──────────────────────────────────────────────
print('Step 8: Feature Engineering — decade, temperature_range, rainfall_category')
df_final = step8_feature_engineering(df_clean)
print(f'  Final shape: {df_final.shape}')
print(f'  Columns: {list(df_final.columns)}')

In [ ]:
# Save final merged dataset
import pathlib
pathlib.Path('data/processed').mkdir(parents=True, exist_ok=True)
df_final.to_csv('data/processed/merged_dataset.csv', index=False)
print('✓ Saved to data/processed/merged_dataset.csv')
df_final.describe().round(2)

---
## Part 2 — Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Crop Yield (Target Variable)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_final['yield_hgha'].dropna(), bins=40, color='#2ecc71', edgecolor='white')
axes[0].set_title('Crop Yield Distribution (hg/ha)')
axes[0].set_xlabel('Yield (hg/ha)')
axes[0].set_ylabel('Count')

# Yield by Crop Type
df_final.groupby('Crop_Type')['yield_hgha'].median().sort_values().plot(
    kind='barh', ax=axes[1], color='#3498db', edgecolor='white')
axes[1].set_title('Median Yield by Crop Type')
axes[1].set_xlabel('Median Yield (hg/ha)')

plt.tight_layout()
plt.savefig('outputs/eda_yield_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation Heatmap
num_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()
corr = df_final[num_cols].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Temperature vs Yield
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pairs = [('avg_temp', 'Temperature (°C)'),
         ('rainfall_mm', 'Rainfall (mm)'),
         ('nitrogen_N', 'Nitrogen (N)')]

for ax, (col, label) in zip(axes, pairs):
    if col in df_final.columns:
        ax.scatter(df_final[col], df_final['yield_hgha'], alpha=0.3, s=10, color='#8e44ad')
        ax.set_xlabel(label)
        ax.set_ylabel('Yield (hg/ha)')
        ax.set_title(f'{label} vs Yield')

plt.tight_layout()
plt.savefig('outputs/eda_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Yield category distribution
df_final['yield_cat'] = pd.cut(
    df_final['yield_hgha'],
    bins=[0, 20000, 50000, float('inf')],
    labels=['Low', 'Medium', 'High']
)

counts = df_final['yield_cat'].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
       colors=['#e74c3c', '#f39c12', '#27ae60'], startangle=140)
ax.set_title('Yield Category Distribution')
plt.savefig('outputs/eda_yield_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print(counts)

---
## Part 3 — ML Models: Regression (Predict yield_hgha)

In [ ]:
from ml_models import (
    prepare_data, get_regression_models, evaluate_regression,
    get_classification_models, evaluate_classification,
    plot_regression_results, plot_classification_results,
    plot_feature_importance, plot_actual_vs_predicted,
    plot_confusion_matrix, FEATURE_COLS, TARGET
)
import joblib

# Drop temp yield_cat column if present
df_ml = df_final.drop(columns=['yield_cat'], errors='ignore')

(X_train, X_test, y_reg_train, y_reg_test,
 y_cls_train, y_cls_test, prep, feature_names) = prepare_data(df_ml)

print(f'Train samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')
print(f'Features used: {len(feature_names)}')
print(f'Features: {feature_names}')

In [ ]:
import numpy as np

reg_results = []
reg_predictions = {}
reg_trained_models = {}

for name, model in get_regression_models().items():
    metrics, y_pred = evaluate_regression(
        model, X_train, X_test, y_reg_train, y_reg_test, name)
    reg_results.append(metrics)
    reg_predictions[name] = y_pred
    reg_trained_models[name] = model
    print(f"  {name:20s}  RMSE={metrics['RMSE']:>12,.1f}  "
          f"MAE={metrics['MAE']:>12,.1f}  R²={metrics['R²']:.4f}  "
          f"CV R²={metrics['CV R² (mean)']:.4f}")

reg_df = pd.DataFrame(reg_results)
display(reg_df)

In [ ]:
plot_regression_results(reg_df)

In [ ]:
# Best regressor — Actual vs Predicted
best_reg_name = reg_df.loc[reg_df['R²'].idxmax(), 'Model']
best_reg_model = reg_trained_models[best_reg_name]
best_reg_preds = reg_predictions[best_reg_name]

print(f'Best Regressor: {best_reg_name} (R² = {reg_df["R²"].max():.4f})')
plot_actual_vs_predicted(y_reg_test, best_reg_preds, best_reg_name)
plot_feature_importance(best_reg_model, feature_names, f'Regression {best_reg_name}')

joblib.dump(best_reg_model, 'models/best_regressor.pkl')
print('✓ Saved → models/best_regressor.pkl')

---
## Part 4 — ML Models: Classification (High / Medium / Low Yield)

In [ ]:
cls_results = []
cls_predictions = {}
cls_trained_models = {}

for name, model in get_classification_models().items():
    metrics, y_pred = evaluate_classification(
        model, X_train, X_test, y_cls_train, y_cls_test, name)
    cls_results.append(metrics)
    cls_predictions[name] = y_pred
    cls_trained_models[name] = model
    print(f"  {name:20s}  Accuracy={metrics['Accuracy']:.4f}  "
          f"F1={metrics['F1 Score (weighted)']:.4f}")

cls_df = pd.DataFrame(cls_results)
display(cls_df)

In [ ]:
plot_classification_results(cls_df)

In [ ]:
from sklearn.metrics import classification_report

best_cls_name = cls_df.loc[cls_df['F1 Score (weighted)'].idxmax(), 'Model']
best_cls_model = cls_trained_models[best_cls_name]
best_cls_preds = cls_predictions[best_cls_name]

print(f'Best Classifier: {best_cls_name} (F1 = {cls_df["F1 Score (weighted)"].max():.4f})')
print('\nClassification Report:')
print(classification_report(y_cls_test, best_cls_preds))
plot_confusion_matrix(y_cls_test, best_cls_preds, best_cls_name)
plot_feature_importance(best_cls_model, feature_names, f'Classification {best_cls_name}')

joblib.dump(best_cls_model, 'models/best_classifier.pkl')
print('✓ Saved → models/best_classifier.pkl')

---
## Part 5 — CNN: Plant Disease Detection
**Architecture:** MobileNetV2 (Transfer Learning) + Custom Classification Head  
**Datasets:** PlantVillage, Rice Leaf Disease, Crop Disease Detection  
**Classes:** Healthy, Bacterial Blight, Leaf Blast, Brown Spot, Tungro, Early Blight, Late Blight, Common Rust

In [ ]:
try:
    import tensorflow as tf
    print(f'TensorFlow version: {tf.__version__}')
    gpus = tf.config.list_physical_devices('GPU')
    print(f'GPUs available: {gpus if gpus else "None (using CPU)"}')
    TF_AVAILABLE = True
except ImportError:
    print('TensorFlow not installed. Run: pip install tensorflow')
    TF_AVAILABLE = False

In [ ]:
if TF_AVAILABLE:
    from cnn_model import (
        generate_synthetic_images, build_data_generators, build_model,
        make_gradcam_heatmap, save_gradcam_image, compute_disease_severity,
        extract_image_features, plot_training_history,
        DATA_DIR, DISEASE_CLASSES, IMG_SIZE, EPOCHS_FROZEN, EPOCHS_FINE
    )
    import pathlib

    # Generate synthetic images (replace with real dataset folders if available)
    print('Generating synthetic plant disease images...')
    generate_synthetic_images(pathlib.Path('data/plant_images'), n_per_class=60)
    print('Done ✓')

In [ ]:
if TF_AVAILABLE:
    train_gen, val_gen = build_data_generators(pathlib.Path('data/plant_images'))
    num_classes = len(train_gen.class_indices)
    class_names = list(train_gen.class_indices.keys())
    print(f'Classes: {class_names}')
    print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)}')

In [ ]:
if TF_AVAILABLE:
    from tensorflow.keras import optimizers, callbacks

    print('Building MobileNetV2 model...')
    model, base = build_model(num_classes, freeze_base=True)
    model.summary()

    cb_list = [
        callbacks.EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
    ]

    print('\n── Phase 1: Feature Extraction (frozen base) ──')
    h1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_FROZEN, callbacks=cb_list
    )
    plot_training_history(h1, 'Phase 1 — Frozen Base')

In [ ]:
if TF_AVAILABLE:
    print('── Phase 2: Fine-Tuning (last 30 layers unfrozen) ──')
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=optimizers.Adam(1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    h2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_FINE, callbacks=cb_list
    )
    plot_training_history(h2, 'Phase 2 — Fine Tuning')

In [ ]:
if TF_AVAILABLE:
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

    val_loss, val_acc = model.evaluate(val_gen, verbose=0)
    print(f'Validation Accuracy: {val_acc:.4f}  |  Validation Loss: {val_loss:.4f}')

    val_gen.reset()
    y_true = val_gen.classes
    y_pred_proba = model.predict(val_gen, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
    ax.set_title('CNN Confusion Matrix — Plant Disease Detection')
    plt.tight_layout()
    plt.savefig('outputs/cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if TF_AVAILABLE:
    # Grad-CAM XAI Visualization
    last_conv = [l.name for l in model.layers if 'conv' in l.name][-1]
    sample_imgs, _ = next(iter(val_gen))
    img_arr = sample_imgs[:1]

    heatmap = make_gradcam_heatmap(img_arr, model, last_conv)
    save_gradcam_image(img_arr, heatmap, pathlib.Path('outputs/gradcam_sample.png'))

    severity = compute_disease_severity(heatmap)
    feats = extract_image_features(model, img_arr, class_names)
    feats['Disease_Severity_Pct'] = severity

    print('Extracted Image Features:')
    feature_df = pd.DataFrame([feats])
    display(feature_df)

In [ ]:
if TF_AVAILABLE:
    model.save('models/plant_disease_cnn.keras')
    print('✓ CNN model saved → models/plant_disease_cnn.keras')

---
## Summary Table

### Final Feature Set (22 Input Features + 1 Target)

| # | Feature | Category | Type | Justification |
|---|---------|----------|------|---------------|
| 1 | Year | Common | Int | Time trend analysis |
| 2 | Country / Region | Common | String | Geographic context |
| 3 | avg_temp (°C) | Common | Float | Primary climate driver |
| 4 | rainfall_mm | Common | Float | Primary crop water source |
| 5 | humidity_pct (%) | Common | Float | Crop moisture requirement |
| 6 | Crop_Type | Common | Categorical | Different crops, different conditions |
| 7 | min_temp (°C) | Climate | Float | Frost damage risk |
| 8 | max_temp (°C) | Climate | Float | Heat stress on crops |
| 9 | wind_speed_kmh | Climate | Float | Affects evaporation |
| 10 | season | Climate | Categorical | Rabi/Kharif affects yield type |
| 11 | sunshine_hours | Climate | Float | Photosynthesis driver |
| 12 | nitrogen_N | Soil | Float | Key soil nutrient |
| 13 | phosphorous_P | Soil | Float | Root development nutrient |
| 14 | potassium_K | Soil | Float | Disease resistance nutrient |
| 15 | soil_pH | Soil | Float | Nutrient availability |
| 16 | soil_moisture_pct (%) | Soil | Float | Water retention |
| 17 | soil_type | Soil | Categorical | Drainage & fertility |
| 18 | fertilizer_kgha | Soil | Float | Human input to yield |
| 19 | AQI | Environmental | Float | Overall pollution level |
| 20 | CO2_ppm | Environmental | Float | Greenhouse gas — photosynthesis |
| 21 | PM25_ugm3 | Environmental | Float | Affects sunlight & plant health |
| 22 | NO2_ppb | Environmental | Float | Acid rain precursor |
| ★ | **yield_hgha** | **TARGET** | Float | **Output to PREDICT** |

In [ ]:
print('=' * 55)
print('  PIPELINE COMPLETE')
print('=' * 55)
print(f'  Final dataset shape  : {df_final.shape}')
print(f'  Best Regressor       : {best_reg_name}')
print(f'  Best Regressor R²    : {reg_df["R²"].max():.4f}')
print(f'  Best Classifier      : {best_cls_name}')
print(f'  Best Classifier F1   : {cls_df["F1 Score (weighted)"].max():.4f}')
print()
print('  Outputs saved to     : outputs/')
print('  Models saved to      : models/')
print('=' * 55)